# 🚀 PhishGuard v2 — Data-Centric AI, TF-IDF Char N-Gram, Fuzzy Typosquatting & LightGBM (Google Colab)
### Arsitektur ML Generasi Lanjutan untuk Mengatasi Bias Dataset

Notebook ini adalah **eksperimen & pipeline training PhishGuard v2** untuk Google Colab yang mengatasi kelemahan model berbasis fitur leksikal dasar (bias `has_https` dan panjang path pada dataset PhiUSIIL):

1. **Data-Centric AI (Augmentasi Terarah + Live Threat Feed):**
   - Mengambil dataset resmi **PhiUSIIL** (235k URL) + opsi menarik feed phishing aktif **OpenPhish** (*Continuous Learning*).
   - Menambahkan ragam URL aman dunia nyata (berita ber-UTM, portal `.ac.id`/`.go.id` HTTP & HTTPS, dokumentasi teknis, path korporasi) serta phishing HTTPS pendek lokal Indonesia & typosquatting **khusus pada data latih** (*Data Test tetap 100% murni PhiUSIIL agar evaluasi jujur*).
2. **Feature Engineering Mendalam (27 Fitur v2):**
   - 20 fitur leksikal dasar + `query_length`, `path_entropy`, `is_official_tld`.
   - **Fuzzy Typosquatting Distance (`typosquat_similarity`):** Mendeteksi kemiripan ejaan + *leetspeak* (`g00gle`, `paypa1`, `bcaa`) terhadap merek resmi.
   - **Character N-Gram TF-IDF (3–5 gram):** Dilatih dengan skema *Out-of-Fold* pada URL tanpa prefix `http://`/`https://` agar fokus mengenali pola domain dan path.
3. **Arsitektur Lebih Tajam:**
   - Membandingkan 3 generasi model: **Logistic Regression (Baseline)** vs **Random Forest v1 (20 Fitur)** vs **LightGBM / HistGradientBoosting (v2)**.
4. **Evaluasi Lengkap & Stress Test Dunia Nyata:**
   - Metrik lengkap (Accuracy, Precision, Recall, F1, ROC-AUC), 5 grafik resolusi 300 DPI, analisis *False Positive & False Negative*, serta uji ketahanan pada 10 URL sulit tanpa bantuan whitelist/aturan *if-else*.

In [ ]:
# [1] Setup Dependensi & Konfigurasi Google Colab
import os
import re
import glob
import math
import json
import zipfile
import warnings
import urllib.request
from collections import Counter
from difflib import SequenceMatcher
from urllib.parse import urlsplit
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")

import joblib
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

try:
    from lightgbm import LGBMClassifier
    USE_LIGHTGBM = True
    V2_ALGO_NAME = "LightGBM (LGBMClassifier)"
except ImportError:
    USE_LIGHTGBM = False
    V2_ALGO_NAME = "HistGradientBoostingClassifier"

print(f"✓ Lingkungan Google Colab Siap | Scikit-Learn: {sklearn.__version__} | Engine v2: {V2_ALGO_NAME}")

In [ ]:
# [2] Akuisisi Dataset PhiUSIIL (UCI) + Live Threat Feed OpenPhish (Continuous Learning)

dataset_df = None
dataset_source = "unknown"

# 1. Cek jika CSV PhiUSIIL sudah tersedia di /content atau Google Drive
local_csvs = glob.glob("/content/**/*.csv", recursive=True)
phiusiil_local = [p for p in local_csvs if "phiusiil" in os.path.basename(p).lower()]
if phiusiil_local:
    print(f"✓ Menemukan dataset lokal/Drive di Colab: {phiusiil_local[0]}")
    dataset_df = pd.read_csv(phiusiil_local[0], usecols=lambda c: c in {"URL", "label"})
    dataset_source = "phiusiil"
else:
    print("ℹ Mengunduh PhiUSIIL URL Dataset resmi dari UCI ML Repository...")
    UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip"
    ZIP_PATH = "phiusiil_dataset.zip"
    EXTRACT_DIR = "dataset_extracted"
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    try:
        if not glob.glob(f"{EXTRACT_DIR}/**/*.csv", recursive=True):
            req = urllib.request.Request(UCI_ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as resp, open(ZIP_PATH, "wb") as out_f:
                out_f.write(resp.read())
            with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
                zip_ref.extractall(EXTRACT_DIR)
        extracted_csv = glob.glob(f"{EXTRACT_DIR}/**/*.csv", recursive=True)
        if extracted_csv:
            dataset_df = pd.read_csv(extracted_csv[0], usecols=lambda c: c in {"URL", "label"})
            dataset_source = "phiusiil"
            print(f"✓ Sukses memuat PhiUSIIL: {len(dataset_df):,} baris data!")
    except Exception as e:
        print(f"⚠️ Gagal mengunduh UCI zip ({e}), menggunakan dataset alternatif...")
        ALT_URL = "https://raw.githubusercontent.com/datasets/phishing-urls/main/data/dataset.csv"
        dataset_df = pd.read_csv(ALT_URL)
        dataset_source = "alternative"

url_col = "URL" if "URL" in dataset_df.columns else dataset_df.columns[0]
label_col = "label" if "label" in dataset_df.columns else dataset_df.columns[1]
dataset_df = dataset_df[[url_col, label_col]].dropna().drop_duplicates(subset=[url_col]).copy()

if dataset_source == "phiusiil" or len(dataset_df) > 200000:
    dataset_df["target"] = (dataset_df[label_col].astype(int) == 0).astype(int)  # 1 = Phishing, 0 = Legitimate
else:
    dataset_df["target"] = (dataset_df[label_col].astype(int) == 1).astype(int)
dataset_df = dataset_df.rename(columns={url_col: "URL"})[["URL", "target"]]

# 2. Opsional: Tarik Live Threat Feed dari OpenPhish (Continuous Learning)
live_phish_urls = []
try:
    req = urllib.request.Request("https://openphish.com/feed.txt", headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        raw_feed = resp.read().decode("utf-8", errors="ignore").splitlines()
    live_phish_urls = [u.strip() for u in raw_feed if u.strip().startswith(("http://", "https://"))][:500]
    print(f"✓ Berhasil menarik {len(live_phish_urls)} URL phishing terbaru dari Live Feed OpenPhish.")
except Exception as e:
    print(f"ℹ Live Feed OpenPhish dilewati ({e}); melanjutkan dengan korpus augmentasi terstruktur.")

print(f"✓ Total URL unik PhiUSIIL siap olah: {len(dataset_df):,} ({dataset_df['target'].mean()*100:.1f}% Phishing)")

In [ ]:
# [3] Feature Engineering v2: 20 Fitur Dasar + Fuzzy Typosquatting + Path/Query Metrics

SUSPICIOUS_KEYWORDS = [
    "login", "verify", "account", "secure", "update", "signin",
    "banking", "password", "wallet", "confirm", "recover", "suspended",
    "webscr", "ebayisapi", "paypal", "credential"
]
INDO_KEYWORDS = [
    "undian", "bansos", "prakerja", "dana", "dana-kaget", "pulsa",
    "kejutan", "hadiah", "bca", "bri", "bni", "mandiri", "gopay",
    "shopee", "linkaja", "kuota", "subsidi", "blt", "ojk", "kominfo",
    "klikbca", "brimo"
]
URL_SHORTENERS = {
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "is.gd", "ow.ly",
    "buff.ly", "rebrand.ly", "cutt.ly", "shorturl.at", "tiny.cc",
    "lnkd.in", "s.id", "rb.gy", "v.gd", "clck.ru", "rotf.lol"
}
SUSPICIOUS_TLDS = {
    "xyz", "top", "tk", "ml", "ga", "cf", "gq", "work", "click",
    "buzz", "fit", "rest", "cam", "kim", "bar", "icu", "club",
    "live", "surf", "monster", "cfd"
}
CONFUSABLES = set("\u0430\u0441\u0435\u043e\u0440\u0455\u0445\u0456\u0458\u0443")

OFFICIAL_BRANDS = {
    "google": ["google.com", "google.co.id", "gmail.com"],
    "bca": ["bca.co.id", "klikbca.com"],
    "bri": ["bri.co.id", "ib.bri.co.id"],
    "mandiri": ["bankmandiri.co.id", "livinbymandiri.id"],
    "bni": ["bni.co.id"],
    "dana": ["dana.id"],
    "gopay": ["gopay.co.id", "gojek.com"],
    "shopee": ["shopee.co.id", "shopeepay.co.id"],
    "tokopedia": ["tokopedia.com"],
    "paypal": ["paypal.com"],
    "microsoft": ["microsoft.com", "live.com", "office.com"],
    "apple": ["apple.com", "icloud.com"],
    "detik": ["detik.com"],
    "kompas": ["kompas.com"],
}
OFFICIAL_DOMAINS_FLAT = {d for ds in OFFICIAL_BRANDS.values() for d in ds}
LEET_TABLE = str.maketrans("01345", "oieas")


def calculate_shannon_entropy(text: str) -> float:
    if not text:
        return 0.0
    n = len(text)
    counts = Counter(text)
    return round(-sum((c / n) * math.log2(c / n) for c in counts.values()), 3)


def strip_scheme(url: str) -> str:
    """Menghilangkan prefix http:// atau https:// agar TF-IDF fokus ke struktur domain dan path."""
    return re.sub(r"^https?://", "", str(url).strip(), flags=re.IGNORECASE)


def compute_brand_fuzzy(host: str, full_lower: str) -> tuple[float, int, int]:
    """Menghitung kemiripan typosquatting (termasuk leetspeak) terhadap merek resmi."""
    if not host:
        return 0.0, 0, 0
    h = host[4:] if host.startswith("www.") else host
    is_official = int(any(h == d or h.endswith("." + d) for d in OFFICIAL_DOMAINS_FLAT))
    if is_official:
        return 0.0, 1, 0

    host_tokens = [t for t in re.split(r"[.\-_]", h) if len(t) >= 3]
    best_sim = 0.0
    brand_abuse = 0
    for brand_key in OFFICIAL_BRANDS:
        if brand_key in full_lower:
            brand_abuse = 1
        blen = len(brand_key)
        for tok in host_tokens:
            if abs(len(tok) - blen) > 3:
                continue
            norm_tok = tok.translate(LEET_TABLE)
            sim = max(
                SequenceMatcher(None, tok, brand_key).ratio(),
                SequenceMatcher(None, norm_tok, brand_key).ratio(),
            )
            if sim > best_sim:
                best_sim = sim
    return round(best_sim, 3), 0, brand_abuse


def extract_v2_features(url_str: str) -> dict:
    raw = str(url_str).strip()
    lowered = raw.lower()
    try:
        parsed = urlsplit(raw if "://" in raw else f"//{raw}")
        host = (parsed.hostname or "").rstrip(".").lower()
        port = parsed.port
        path = parsed.path or ""
        query = parsed.query or ""
    except Exception:
        host, port, path, query = "", None, "", ""

    parts = host.split(".")
    is_ip = 1 if len(parts) == 4 and all(p.isdigit() and 0 <= int(p) <= 255 for p in parts) else 0
    subdomain_count = max(0, len([p for p in parts if p]) - 2)
    susp_hits = sum(lowered.count(kw) for kw in SUSPICIOUS_KEYWORDS)
    indo_hits = sum(lowered.count(kw) for kw in INDO_KEYWORDS)
    has_homograph = 1 if ("xn--" in host or any(c in CONFUSABLES for c in host)) else 0
    tld = parts[-1] if parts else ""
    is_suspicious_tld = 1 if tld in SUSPICIOUS_TLDS else 0
    is_official_tld = 1 if (host.endswith(".ac.id") or host.endswith(".go.id") or tld in {"edu", "gov"}) else 0
    is_shortener = 1 if any(host == s or host.endswith("." + s) for s in URL_SHORTENERS) else 0
    has_port = 1 if port is not None else 0
    host_entropy = calculate_shannon_entropy(host)
    path_entropy = calculate_shannon_entropy(path)
    typosquat_sim, is_official_brand, has_brand_abuse = compute_brand_fuzzy(host, lowered)

    return {
        # 20 Fitur Leksikal v1
        "url_length": len(raw),
        "hostname_length": len(host),
        "path_length": len(path),
        "count_dot": raw.count("."),
        "count_hyphen": raw.count("-"),
        "count_at": raw.count("@"),
        "count_question": raw.count("?"),
        "count_equal": raw.count("="),
        "count_slash": raw.count("/"),
        "count_digits": sum(c.isdigit() for c in raw),
        "count_subdomains": subdomain_count,
        "has_https": 1 if lowered.startswith("https://") else 0,
        "has_ip": is_ip,
        "has_port": has_port,
        "shannon_entropy": host_entropy,
        "has_homograph": has_homograph,
        "is_suspicious_tld": is_suspicious_tld,
        "has_url_shortener": is_shortener,
        "suspicious_words": susp_hits,
        "indonesian_keywords": indo_hits,
        # 6 Fitur Tambahan v2
        "query_length": len(query),
        "path_entropy": path_entropy,
        "is_official_tld": is_official_tld,
        "typosquat_similarity": typosquat_sim,
        "is_official_brand": is_official_brand,
        "has_brand_abuse": has_brand_abuse,
    }


V1_FEATURE_COLS = [
    "url_length", "hostname_length", "path_length", "count_dot", "count_hyphen",
    "count_at", "count_question", "count_equal", "count_slash", "count_digits",
    "count_subdomains", "has_https", "has_ip", "has_port", "shannon_entropy",
    "has_homograph", "is_suspicious_tld", "has_url_shortener", "suspicious_words",
    "indonesian_keywords"
]
V2_TABULAR_COLS = V1_FEATURE_COLS + [
    "query_length", "path_entropy", "is_official_tld",
    "typosquat_similarity", "is_official_brand", "has_brand_abuse"
]
V2_ALL_COLS = V2_TABULAR_COLS + ["tfidf_char_prob"]
print(f"✓ Konfigurasi Fitur Siap: v1 ({len(V1_FEATURE_COLS)} fitur) -> v2 ({len(V2_ALL_COLS)} fitur termasuk TF-IDF & Fuzzy)")

In [ ]:
# [4] Stratified Split & Data-Centric Augmentation (Khusus Data Latih — Anti-Leakage)

# 1. Partisi dataset asli PhiUSIIL lebih dulu agar Test Set 100% murni tanpa data augmentasi
train_df, temp_df = train_test_split(dataset_df, test_size=0.30, random_state=42, stratify=dataset_df["target"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["target"])

# 2. Bangun korpus Data-Centric Augmentation proporsional dengan skala train_df
rng = np.random.default_rng(42)
aug_rows = []

domains_edu = ["univ-nusantara", "institut-teknologi", "poltek-bandung", "kampus-merdeka", "akademi-informatika", "stmi-jakarta", "unpad-news", "ugm-press"]
domains_doc = ["readthedocs.io", "kernel.org", "mozilla.org", "apache.org", "w3.org", "debian.org", "rust-lang.org", "golang.dev", "kubernetes.io"]
domains_news = ["tempo.co", "kumparan.com", "antaranews.com", "cnnindonesia.com", "republika.co.id", "sindonews.com", "idntimes.com"]
paths_a = ["library", "reference", "guide", "tutorial", "article", "archive", "modul", "akademik", "kurikulum", "berita", "tekno", "riset", "sistem"]
paths_b = ["overview", "getting-started", "v2", "index", "detail", "panduan", "pengumuman", "dokumentasi", "chapter-1", "release-notes"]
files = ["manual.html", "spec.html", "view.php", "guide.pdf", "overview.md", "report.html", "index.jsp"]

# Sesuaikan ukuran augmentasi dengan ukuran data latih (~10% dari train_df agar seimbang)
n_legit_aug = max(1500, min(18000, int(len(train_df) * 0.10)))
n_phish_aug = max(1200, min(14000, int(len(train_df) * 0.08)))

for i in range(n_legit_aug):
    mode = i % 4
    pa, pb = rng.choice(paths_a), rng.choice(paths_b)
    fn = rng.choice(files)
    if mode == 0:  # Portal kampus/pemerintah HTTP & HTTPS
        sch = "http" if i % 2 == 0 else "https"
        sub = rng.choice(["portal", "siakad", "elearning", "simak", "library", "repo"])
        u = f"{sch}://{sub}.{rng.choice(domains_edu)}.{rng.choice(['ac.id', 'go.id', 'edu'])}/{pa}/{pb}/{i % 999}"
    elif mode == 1:  # Dokumentasi teknis dengan versi, titik, .html & fragment #
        sub = rng.choice(["docs", "api", "wiki", "manual", "dev"])
        u = f"https://{sub}.{rng.choice(domains_doc)}/{(i % 5) + 1}.{i % 12}/{pa}/{pb}/{fn}#{pa}-{pb}"
    elif mode == 2:  # Portal berita panjang dengan parameter pelacakan UTM
        u = f"https://www.{rng.choice(domains_news)}/{pa}/read/2026/{(i % 12) + 1:02d}/{10000 + i}/{pb}?utm_source=share&utm_medium=social&ref=home"
    else:  # Situs perusahaan/e-commerce resmi ber-path dalam
        dom = rng.choice(["bca.co.id", "bankmandiri.co.id", "tokopedia.com", "bukalapak.com", "traveloka.com", "gojek.com"])
        u = f"https://www.{dom}/id/{pa}/{pb}/{1000 + i}?lang=id"
    aug_rows.append((u, 0))

phish_lures = ["dana-kaget", "bansos", "prakerja", "brimo", "klikbca", "blt", "hadiah", "undian", "subsidi", "ojk-blokir", "mybca-aktivasi"]
phish_brands = ["bcaa", "paypa1", "g00gle", "mandlri", "sh0pee", "app1e", "m1crosoft", "netf1ix", "te1egram", "whats4pp"]
phish_actions = ["login", "verify", "update", "signin", "confirm", "daftar", "klaim", "ambil", "auth", "secure"]
bad_tlds = ["xyz", "top", "click", "cfd", "tk", "icu", "work", "buzz"]

for i in range(n_phish_aug):
    mode = i % 3
    act = rng.choice(phish_actions)
    if mode == 0:
        u = f"https://{rng.choice(phish_lures)}-{rng.choice(['resmi', 'gratis', '2026', 'kilat'])}-{i % 900}.{rng.choice(bad_tlds)}/{act}"
    elif mode == 1:
        u = f"https://{rng.choice(phish_brands)}-{rng.choice(['security', 'support', 'alert', 'verify', 'center', 'online'])}-{i % 500}.{rng.choice(['com', 'net', 'xyz', 'top'])}/{act}"
    else:
        u = f"https://xn--{rng.choice(['pple', 'gogle', 'bca', 'bri'])}-{10 + i % 80}a.com/{act}"
    aug_rows.append((u, 1))

# Gabungkan URL dari Live Feed OpenPhish jika berhasil ditarik (hindari duplikasi dengan test_df)
test_url_set = set(test_df["URL"])
for live_u in live_phish_urls:
    if live_u not in test_url_set:
        aug_rows.append((live_u, 1))

aug_df = pd.DataFrame(aug_rows, columns=["URL", "target"])
train_aug_df = pd.concat([train_df, aug_df], ignore_index=True)

# Bobot 3.0x pada sampel augmentasi agar pola dunia nyata tidak tenggelam oleh bias homepage PhiUSIIL
aug_weight = 3.0
sample_weights_v2 = np.concatenate([np.ones(len(train_df)), np.full(len(aug_df), aug_weight)])

print("=" * 65)
print("✓ Data Splitting & Data-Centric Augmentation Selesai:")
print(f"  - Train PhiUSIIL Asli  : {len(train_df):,} baris")
print(f"  - Augmentasi Latih     : {len(aug_df):,} baris (Legit ber-path/HTTP + Phishing HTTPS Indo + OpenPhish)")
print(f"  - Total Train v2       : {len(train_aug_df):,} baris")
print(f"  - Validation Set Murni : {len(val_df):,} baris")
print(f"  - Test Set Murni       : {len(test_df):,} baris (100% PhiUSIIL tanpa sentuhan augmentasi)")
print("=" * 65)

In [ ]:
# [5] Ekstraksi Fitur Paralel & Out-of-Fold Char N-Gram TF-IDF (3-5 gram)

workers = os.cpu_count() or 2

def batch_extract(url_series):
    urls_list = url_series.astype(str).tolist()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        rows = list(tqdm(ex.map(extract_v2_features, urls_list), total=len(urls_list)))
    return pd.DataFrame(rows)[V2_TABULAR_COLS]

print("Mengekstrak 26 fitur tabular + fuzzy untuk Train v2, Validation, dan Test...")
X_train_v2 = batch_extract(train_aug_df["URL"])
X_val_v2 = batch_extract(val_df["URL"])
X_test_v2 = batch_extract(test_df["URL"])

y_train_v2 = train_aug_df["target"].values
y_val = val_df["target"].values
y_test = test_df["target"].values

# Fitur v1 (20 fitur murni pada train_df asli tanpa augmentasi) untuk komparasi adil v1 vs v2
X_train_v1 = X_train_v2.iloc[:len(train_df)][V1_FEATURE_COLS].copy()
y_train_v1 = train_df["target"].values
X_test_v1 = X_test_v2[V1_FEATURE_COLS].copy()

# Out-of-Fold Char N-Gram TF-IDF pada URL tanpa prefix http:// / https://
print("Melatih Char N-Gram TF-IDF (3-5 gram) Out-of-Fold...")
train_stripped = train_aug_df["URL"].map(strip_scheme)
val_stripped = val_df["URL"].map(strip_scheme)
test_stripped = test_df["URL"].map(strip_scheme)

oof_tfidf_prob = np.zeros(len(train_aug_df))
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for tr_idx, va_idx in skf.split(train_stripped, y_train_v2):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=5000, sublinear_tf=True)
    clf_tf = LogisticRegression(max_iter=300, C=2.0)
    clf_tf.fit(vec.fit_transform(train_stripped.iloc[tr_idx]), y_train_v2[tr_idx], sample_weight=sample_weights_v2[tr_idx])
    oof_tfidf_prob[va_idx] = clf_tf.predict_proba(vec.transform(train_stripped.iloc[va_idx]))[:, 1]

full_tfidf_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=5000, sublinear_tf=True)
full_tfidf_clf = LogisticRegression(max_iter=300, C=2.0)
full_tfidf_clf.fit(full_tfidf_vec.fit_transform(train_stripped), y_train_v2, sample_weight=sample_weights_v2)

X_train_v2["tfidf_char_prob"] = oof_tfidf_prob
X_val_v2["tfidf_char_prob"] = full_tfidf_clf.predict_proba(full_tfidf_vec.transform(val_stripped))[:, 1]
X_test_v2["tfidf_char_prob"] = full_tfidf_clf.predict_proba(full_tfidf_vec.transform(test_stripped))[:, 1]

print("✓ Ekstraksi fitur v2 + TF-IDF Out-of-Fold selesai!")

In [ ]:
# [6] Pelatihan & Komparasi 3 Generasi Model: Baseline LR vs RF v1 vs PhishGuard v2

# 1. Baseline: Logistic Regression (20 Fitur v1 pada PhiUSIIL murni)
print("[1/3] Melatih Baseline: Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr_model.fit(X_train_v1, y_train_v1)

# 2. PhishGuard v1: Random Forest 150 Trees (20 Fitur v1 pada PhiUSIIL murni)
print("[2/3] Melatih PhishGuard v1: Random Forest (150 Trees)...")
rf_v1_model = RandomForestClassifier(
    n_estimators=150, criterion="gini", max_features="sqrt",
    min_samples_split=3, class_weight="balanced", random_state=42, n_jobs=-1
)
rf_v1_model.fit(X_train_v1, y_train_v1)

# 3. PhishGuard v2: LightGBM / HistGradientBoosting (27 Fitur v2 + Data-Centric Augmentation)
print(f"[3/3] Melatih PhishGuard v2: {V2_ALGO_NAME}...")
if USE_LIGHTGBM:
    v2_model = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.07,
        num_leaves=63,
        subsample_for_bin=50000,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
else:
    v2_model = HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.08,
        max_leaf_nodes=63,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=42
    )

v2_model.fit(X_train_v2, y_train_v2, sample_weight=sample_weights_v2)
print("✓ Ketiga model selesai dilatih!")

In [ ]:
# [7] Evaluasi Ilmiah & 4 Grafik Resolusi Tinggi (300 DPI) pada Held-Out Test Set

def compute_metrics(model, X_eval, y_true):
    prob = model.predict_proba(X_eval)[:, 1]
    pred = (prob >= 0.50).astype(int)
    cm_mat = confusion_matrix(y_true, pred)
    tn_c, fp_c, fn_c, tp_c = cm_mat.ravel()
    return {
        "prob": prob,
        "pred": pred,
        "cm": cm_mat,
        "tn": int(tn_c), "fp": int(fp_c), "fn": int(fn_c), "tp": int(tp_c),
        "fpr": fp_c / (fp_c + tn_c) if (fp_c + tn_c) > 0 else 0.0,
        "fnr": fn_c / (fn_c + tp_c) if (fn_c + tp_c) > 0 else 0.0,
        "acc": accuracy_score(y_true, pred),
        "prec": precision_score(y_true, pred, zero_division=0),
        "rec": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "auc": roc_auc_score(y_true, prob),
    }

res_lr = compute_metrics(lr_model, X_test_v1, y_test)
res_v1 = compute_metrics(rf_v1_model, X_test_v1, y_test)
res_v2 = compute_metrics(v2_model, X_test_v2, y_test)

print("=" * 84)
print("          KOMPARASI PERFORMA PADA CLEAN HELD-OUT TEST SET PHIUSIIL")
print("=" * 84)
print(f"{'Model':<38} | {'Acc':>7} | {'Prec':>7} | {'Rec':>7} | {'F1':>7} | {'AUC':>7}")
print("-" * 84)
for label_m, r in [
    ("Logistic Regression (Baseline)", res_lr),
    ("PhishGuard v1 (RF 20 Fitur)", res_v1),
    (f"PhishGuard v2 ({V2_ALGO_NAME[:18]})", res_v2),
]:
    print(f"{label_m:<38} | {r['acc']*100:6.2f}% | {r['prec']*100:6.2f}% | {r['rec']*100:6.2f}% | {r['f1']*100:6.2f}% | {r['auc']:7.4f}")
print("=" * 84)
print("\nClassification Report — PhishGuard v2:")
print(classification_report(y_test, res_v2["pred"], target_names=["Legitimate (0)", "Phishing (1)"], digits=4))

# Grafik 1: Perbandingan 3 Generasi Model
metric_keys = ["acc", "prec", "rec", "f1", "auc"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
x_pos = np.arange(len(metric_labels))
w = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x_pos - w, [res_lr[k] * 100 for k in metric_keys], w, label="Logistic Regression (Baseline)", color="#94A3B8")
plt.bar(x_pos, [res_v1[k] * 100 for k in metric_keys], w, label="PhishGuard v1 (RF 20 Fitur)", color="#3B82F6")
bars_v2 = plt.bar(x_pos + w, [res_v2[k] * 100 for k in metric_keys], w, label="PhishGuard v2 (LightGBM/HGB + TF-IDF + Fuzzy)", color="#0F766E")
for b in bars_v2:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width() / 2, h + 0.3, f"{h:.2f}%", ha="center", va="bottom", fontsize=8, fontweight="bold", color="#0F766E")
plt.xticks(x_pos, metric_labels, fontweight="bold")
plt.ylabel("Skor (%)")
plt.ylim(75, 104)
plt.title("Perbandingan 3 Generasi Model pada Data Uji PhiUSIIL", fontsize=12, fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("v2_model_comparison.png", dpi=300)
plt.show()

# Grafik 2: Confusion Matrix PhishGuard v2
plt.figure(figsize=(6, 5))
sns.heatmap(
    res_v2["cm"], annot=True, fmt="d", cmap="Teal" if "Teal" in plt.colormaps() else "YlGnBu",
    xticklabels=["Legitimate (0)", "Phishing (1)"], yticklabels=["Legitimate (0)", "Phishing (1)"]
)
plt.title(f"Confusion Matrix PhishGuard v2 ({len(y_test):,} Sampel Uji)", fontweight="bold")
plt.xlabel("Prediksi Model")
plt.ylabel("Label Sebenarnya")
plt.tight_layout()
plt.savefig("v2_confusion_matrix.png", dpi=300)
plt.show()

# Grafik 3: Kurva ROC-AUC (Baseline vs v1 vs v2)
plt.figure(figsize=(7, 5.5))
for lbl, r, col, ls in [
    ("PhishGuard v2", res_v2, "#0F766E", "-"),
    ("PhishGuard v1 (RF)", res_v1, "#1D4ED8", "--"),
    ("Baseline (LogReg)", res_lr, "#F59E0B", "-."),
]:
    fpr_c, tpr_c, _ = roc_curve(y_test, r["prob"])
    plt.plot(fpr_c, tpr_c, color=col, lw=2.2, linestyle=ls, label=f"{lbl} (AUC = {r['auc']:.4f})")
plt.plot([0, 1], [0, 1], color="#9CA3AF", lw=1.2, linestyle=":", label="Random (AUC = 0.5000)")
plt.title("Kurva ROC-AUC: Baseline vs v1 vs PhishGuard v2", fontsize=12, fontweight="bold")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.grid(linestyle="--", alpha=0.4)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("v2_roc_curve.png", dpi=300)
plt.show()

# Grafik 4: Permutation Feature Importance PhishGuard v2
n_imp = min(3000, len(X_test_v2))
perm = permutation_importance(v2_model, X_test_v2.iloc[:n_imp], y_test[:n_imp], n_repeats=3, random_state=42, n_jobs=1)
v2_importances = pd.Series(perm.importances_mean, index=V2_ALL_COLS).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
v2_importances.tail(20).plot(kind="barh", color="#0F766E")
plt.title("Top 20 Feature Importance PhishGuard v2 (Permutation Importance)", fontsize=12, fontweight="bold")
plt.xlabel("Mean Accuracy Decrease")
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("v2_feature_importance.png", dpi=300)
plt.show()

In [ ]:
# [8] Analisis Mendalam False Positive (FP) & False Negative (FN) PhishGuard v2

eval_df = X_test_v2.copy()
eval_df["url"] = test_df["URL"].values
eval_df["actual"] = y_test
eval_df["predicted"] = res_v2["pred"]
eval_df["phishing_prob"] = np.round(res_v2["prob"], 4)

tn_df = eval_df[(eval_df["actual"] == 0) & (eval_df["predicted"] == 0)]
fp_df = eval_df[(eval_df["actual"] == 0) & (eval_df["predicted"] == 1)].sort_values("phishing_prob", ascending=False)
fn_df = eval_df[(eval_df["actual"] == 1) & (eval_df["predicted"] == 0)].sort_values("phishing_prob", ascending=True)
tp_df = eval_df[(eval_df["actual"] == 1) & (eval_df["predicted"] == 1)]

print("=" * 75)
print("        ANALISIS KESALAHAN KLASIFIKASI PHISHGUARD v2 (ERROR ANALYSIS)")
print("=" * 75)
print(f"False Positive (FP): {len(fp_df):,} URL aman tertandai Phishing (FPR = {res_v2['fpr']*100:.2f}%)")
print(f"False Negative (FN): {len(fn_df):,} URL phishing lolos sebagai Aman (FNR = {res_v2['fnr']*100:.2f}%)")

profile_cols = ["url_length", "path_length", "shannon_entropy", "typosquat_similarity", "tfidf_char_prob", "has_https"]
summary_profiles = pd.DataFrame({
    "TN (Aman Benar)": tn_df[profile_cols].mean(),
    "FP (Salah Blokir)": fp_df[profile_cols].mean() if len(fp_df) > 0 else pd.Series(0, index=profile_cols),
    "FN (Phishing Lolos)": fn_df[profile_cols].mean() if len(fn_df) > 0 else pd.Series(0, index=profile_cols),
    "TP (Phishing Benar)": tp_df[profile_cols].mean(),
}).round(3)
print("\nRata-Rata Fitur Utama per Kuadran:")
print(summary_profiles.to_string())

print("\n--- Contoh 10 Sampel FALSE POSITIVE ---")
if len(fp_df) == 0:
    print("Tidak ada False Positive.")
else:
    for i, (_, row) in enumerate(fp_df.head(10).iterrows(), 1):
        u_disp = row["url"][:90] + ("..." if len(row["url"]) > 90 else "")
        print(f"{i:2d}. [Prob={row['phishing_prob']:.4f} | tfidf={row['tfidf_char_prob']:.3f}, path={int(row['path_length'])}] {u_disp}")

print("\n--- Contoh 10 Sampel FALSE NEGATIVE ---")
if len(fn_df) == 0:
    print("Tidak ada False Negative.")
else:
    for i, (_, row) in enumerate(fn_df.head(10).iterrows(), 1):
        u_disp = row["url"][:90] + ("..." if len(row["url"]) > 90 else "")
        print(f"{i:2d}. [Prob={row['phishing_prob']:.4f} | tfidf={row['tfidf_char_prob']:.3f}, path={int(row['path_length'])}] {u_disp}")

# Grafik 5: Visualisasi Error Analysis v2
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
groups = ["TN\n(Aman Benar)", "FP\n(Salah Blokir)", "FN\n(Lolos)", "TP\n(Phishing Benar)"]
means_tfidf = [
    tn_df["tfidf_char_prob"].mean(),
    fp_df["tfidf_char_prob"].mean() if len(fp_df) > 0 else 0,
    fn_df["tfidf_char_prob"].mean() if len(fn_df) > 0 else 0,
    tp_df["tfidf_char_prob"].mean(),
]
axes[0].bar(groups, means_tfidf, color=["#10B981", "#F59E0B", "#EF4444", "#0F766E"])
axes[0].set_ylabel("Rata-rata Skor TF-IDF Char N-Gram")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Kontribusi TF-IDF Char N-Gram per Kuadran Prediksi", fontweight="bold")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

if len(fp_df) > 0:
    axes[1].hist(fp_df["phishing_prob"], bins=15, alpha=0.7, color="#F59E0B", label=f"False Positive (n={len(fp_df)})")
if len(fn_df) > 0:
    axes[1].hist(fn_df["phishing_prob"], bins=15, alpha=0.7, color="#DC2626", label=f"False Negative (n={len(fn_df)})")
axes[1].axvline(0.50, color="#111827", linestyle="--", lw=1.8, label="Threshold (0.50)")
axes[1].set_xlim(0.0, 1.0)
axes[1].set_xlabel("Probabilitas Prediksi Model v2")
axes[1].set_ylabel("Jumlah Sampel")
axes[1].set_title("Distribusi Probabilitas pada Kasus FP & FN", fontweight="bold")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
axes[1].legend()

plt.tight_layout()
plt.savefig("v2_error_analysis.png", dpi=300)
plt.show()

In [ ]:
# [9] Stress Test Dunia Nyata: 10 URL Sulit (Murni Output ML, Tanpa Whitelist / Aturan Hardcode)

unseen_hard_urls = [
    ("https://www.liputan6.com/tekno/read/5812345/tips-keamanan-digital-anti-penipuan?utm_source=fb&utm_medium=social", 0, "Aman: Berita path panjang + UTM"),
    ("http://portal.akademik-stikom.ac.id/mhs/krs/cetak", 0, "Aman: HTTP kampus + path"),
    ("https://www.bri.co.id/id/individu/produk-simpanan/britama?ref=homepage_banner", 0, "Aman: Bank resmi path panjang"),
    ("https://docs.python.org/3/library/urllib.parse.html#module-urllib.parse", 0, "Aman: Dokumentasi panjang"),
    ("https://dana-kaget-gratis-2026.xyz/", 1, "Phishing: HTTPS pendek + Indo lure"),
    ("https://cek-penerima-blt-kemensos.top/ambil", 1, "Phishing: HTTPS + Bansos Indo"),
    ("https://mybca-aktivasi-ulang.com/login", 1, "Phishing: HTTPS + Brand BCA palsu"),
    ("https://g00gle-security-alert.com/verify", 1, "Phishing: HTTPS + Typosquatting g00gle"),
    ("https://paypa1-resolution-center.net/signin", 1, "Phishing: HTTPS + Typosquatting paypa1"),
    ("https://xn--pple-43d.com/account", 1, "Phishing: HTTPS + Homograph Apple"),
]

print("=" * 98)
print("       STRESS TEST MURNI MODEL ML (TANPA WHITELIST / ATURAN IF-ELSE) — 10 URL SULIT")
print("=" * 98)
print(f"{'Deskripsi':<38} | {'Label':<5} | {'Model v1 (RF)':<16} | {'Model v2 (New)':<16}")
print("-" * 98)

v1_hits, v2_hits = 0, 0
for u_str, true_lbl, desc in unseen_hard_urls:
    f_dict = extract_v2_features(u_str)

    # Prediksi v1 (Raw RF 20 Fitur)
    x_v1 = pd.DataFrame([[f_dict[c] for c in V1_FEATURE_COLS]], columns=V1_FEATURE_COLS)
    p_v1 = float(rf_v1_model.predict_proba(x_v1)[0][1])
    ok_v1 = int(p_v1 >= 0.50) == true_lbl
    v1_hits += int(ok_v1)

    # Prediksi v2 (Tabular + Fuzzy + TF-IDF Char N-Gram + LightGBM/HGB)
    f_dict["tfidf_char_prob"] = float(full_tfidf_clf.predict_proba(full_tfidf_vec.transform([strip_scheme(u_str)]))[0][1])
    x_v2 = pd.DataFrame([[f_dict[c] for c in V2_ALL_COLS]], columns=V2_ALL_COLS)
    p_v2 = float(v2_model.predict_proba(x_v2)[0][1])
    ok_v2 = int(p_v2 >= 0.50) == true_lbl
    v2_hits += int(ok_v2)

    lbl_txt = "PHISH" if true_lbl == 1 else "AMAN "
    print(f"{desc:<38} | {lbl_txt} | {p_v1*100:5.1f}% {'OK' if ok_v1 else 'SALAH':<6} | {p_v2*100:5.1f}% {'OK' if ok_v2 else 'SALAH':<6}")

print("-" * 98)
print(f"Total Akurasi Murni di 10 URL Sulit -> Model v1: {v1_hits}/10 ({v1_hits*10}%) | Model v2: {v2_hits}/10 ({v2_hits*10}%)")
print("=" * 98)

In [ ]:
# [10] Ekspor Artefak PhishGuard v2 & Auto-Download Google Colab

# 1. Simpan model_phishing_v2.pkl (lengkap dengan pipeline TF-IDF dan daftar 27 fitur)
v2_bundle = {
    "model": v2_model,
    "tfidf_vectorizer": full_tfidf_vec,
    "tfidf_classifier": full_tfidf_clf,
    "feature_names": V2_ALL_COLS,
    "tabular_feature_names": V2_TABULAR_COLS,
    "threshold": 0.50,
    "algorithm": V2_ALGO_NAME,
    "metadata": {
        "version": "2.0-data-centric",
        "dataset": "PhiUSIIL + Data-Centric Augmentation + OpenPhish",
        "features_count": len(V2_ALL_COLS),
        "live_feed_samples_added": len(live_phish_urls),
        "training_environment": "Google Colab"
    }
}
joblib.dump(v2_bundle, "model_phishing_v2.pkl")
print(f"✓ Sukses mengekspor model_phishing_v2.pkl ({os.path.getsize('model_phishing_v2.pkl') / 1e6:.2f} MB)")

# 2. Simpan metrics_v2.json
feature_rankings = [
    {"feature": feat, "importance": round(float(imp), 4), "rank": i + 1}
    for i, (feat, imp) in enumerate(v2_importances.sort_values(ascending=False).items())
]

metrics_v2 = {
    "dataset": {
        "name": "PhiUSIIL + Data-Centric Augmentation",
        "phiusiil_unique_samples": len(dataset_df),
        "augmented_train_samples": len(aug_df),
        "live_openphish_added": len(live_phish_urls),
        "features_count": len(V2_ALL_COLS),
    },
    "model": {
        "algorithm": V2_ALGO_NAME,
        "threshold": 0.50,
    },
    "test_metrics_phiusiil": {
        "accuracy": round(float(res_v2["acc"]), 4),
        "precision": round(float(res_v2["prec"]), 4),
        "recall": round(float(res_v2["rec"]), 4),
        "f1_score": round(float(res_v2["f1"]), 4),
        "roc_auc": round(float(res_v2["auc"]), 4),
    },
    "v1_test_metrics_phiusiil": {
        "accuracy": round(float(res_v1["acc"]), 4),
        "precision": round(float(res_v1["prec"]), 4),
        "recall": round(float(res_v1["rec"]), 4),
        "f1_score": round(float(res_v1["f1"]), 4),
        "roc_auc": round(float(res_v1["auc"]), 4),
    },
    "baseline_metrics_phiusiil": {
        "accuracy": round(float(res_lr["acc"]), 4),
        "precision": round(float(res_lr["prec"]), 4),
        "recall": round(float(res_lr["rec"]), 4),
        "f1_score": round(float(res_lr["f1"]), 4),
        "roc_auc": round(float(res_lr["auc"]), 4),
    },
    "stress_test_10_hard_urls": {
        "v1_correct": v1_hits,
        "v2_correct": v2_hits,
        "total": len(unseen_hard_urls),
    },
    "confusion_matrix": {
        "true_negatives": res_v2["tn"],
        "false_positives": res_v2["fp"],
        "false_negatives": res_v2["fn"],
        "true_positives": res_v2["tp"],
        "false_positive_rate": round(float(res_v2["fpr"]), 6),
        "false_negative_rate": round(float(res_v2["fnr"]), 6),
    },
    "feature_importance": feature_rankings
}

with open("metrics_v2.json", "w", encoding="utf-8") as f:
    json.dump(metrics_v2, f, indent=2)
print("✓ Sukses mengekspor metrics_v2.json")

# 3. Kemas seluruh 5 grafik v2 ke ZIP
plot_files = [
    "v2_model_comparison.png",
    "v2_confusion_matrix.png",
    "v2_roc_curve.png",
    "v2_feature_importance.png",
    "v2_error_analysis.png",
]
with zipfile.ZipFile("phishguard_v2_evaluation_plots.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for pf in plot_files:
        if os.path.isfile(pf):
            zf.write(pf)
print("✓ Sukses mengemas 5 grafik evaluasi ke phishguard_v2_evaluation_plots.zip")

# 4. Auto-Download di Google Colab
try:
    from google.colab import files
    print("Mengunduh artefak PhishGuard v2 dari Google Colab...")
    files.download("model_phishing_v2.pkl")
    files.download("metrics_v2.json")
    files.download("phishguard_v2_evaluation_plots.zip")
except ImportError:
    print("ℹ Di luar Google Colab: semua file tersedia di direktori kerja saat ini.")